# Study 885 — Ultra-Short Credit Pickup 💵

**Do ultra-short investment-grade credit ETFs (JPST / ICSH / MINT) pay you a real,
near-riskless pickup over plain T-bills (BIL / SHV)?**

The pitch is mechanical: hold ~AA-/A short-maturity IG corporates and ABS instead of
pure bills, and you are paid a small **spread over cash** for a *sliver* of credit and
duration risk. If that pickup is a genuine structural premium, the credit sleeve should
earn a **higher excess-of-bills Sharpe** than bills — better reward per unit of risk —
while drawing down only marginally more. We test it on the live, fee-paying tape
(2017-05-22 → 2026-06-30, 2289 common days) and stay honest about 2020 & 2022.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `22e1cddb739d`);
the live cells run the fast synthetic control. Young ETFs → short live history, named on
the Signal axis.*


## 1. The idea in one line

Bills (BIL) pay the risk-free rate for essentially zero risk. Ultra-short credit (JPST/ICSH/MINT) buys slightly riskier paper — short IG corporates, a touch of ABS, ~0.3–0.9y duration — and pockets the **spread**. The hope: a boring, near-riskless *carry* you collect just by parking cash one notch up the risk ladder.

In [1]:
R = {'fingerprint': '22e1cddb739d', 'asof': '2026-06-30', 'n_days': 2289, 'start': '2017-05-22', 'end': '2026-06-30', 'years': 9.08, 'jpst_sharpe': 0.61, 'icsh_sharpe': 0.54, 'mint_sharpe': 0.4, 'shv_sharpe': 0.11, 'jpst_ret': 3.0, 'icsh_ret': 2.94, 'mint_ret': 2.81, 'bil_ret': 2.41, 'shv_ret': 2.44, 'jpst_vol': 0.93, 'icsh_vol': 0.97, 'mint_vol': 0.98, 'bil_vol': 0.25, 'shv_vol': 0.27, 'sleeve_bps': 49.8, 'sleeve_t': 1.3, 'sleeve_sharpe': 0.62, 'sleeve_lags': 8, 'jpst_bps': 57.9, 'jpst_pt': 1.6, 'icsh_bps': 52.0, 'icsh_pt': 1.46, 'mint_bps': 39.5, 'mint_pt': 0.84, 'shv_bps': 2.8, 'ci_lo': -0.26, 'ci_hi': 2.21, 'ci_fracneg': 0.091, 'ci_block': 13, 'early_bps': 49.2, 'early_t': 3.21, 'early_n': 406, 'late_bps': 49.9, 'late_t': 1.08, 'late_n': 1883, 'welch_t': -0.02, 'bil_dd': -0.21, 'jpst_dd': -3.28, 'icsh_dd': -3.94, 'mint_dd': -4.62, 'y2022_bil': 1.4, 'y2022_jpst': 1.14, 'y2022_icsh': 0.96, 'y2022_mint': -1.01, 'covid_bil': 0.26, 'covid_jpst': -2.75, 'covid_icsh': -3.55, 'covid_mint': -4.34, 'mint_long_bps': 77.0, 'mint_long_t': 2.82, 'mint_long_sharpe': 0.89, 'mint_long_cilo': 0.18, 'mint_long_cihi': 1.92, 'mint_long_n': 4177, 'mint_long_years': 16.6, 'mint_long_early_t': 7.44, 'mint_long_late_t': 0.73, 'cost1_net': 47.8, 'cost1_sharpe': 0.59, 'cost2_net': 45.8, 'cost5_net': 39.8, 'syn_null_tmean': 0.61, 'syn_null_fire': '0/12', 'syn_plant': 120.0, 'syn_plant_tmean': 3.51, 'syn_plant_fire': '11/12', 'syn_plant_recovered': 120.0}
print('sleeve pickup over BIL : %+.1f bps/yr  (HAC t = %+.2f)'
      % (R['sleeve_bps'], R['sleeve_t']))
print('excess-of-BIL Sharpe   : JPST %+.2f  ICSH %+.2f  MINT %+.2f  vs  SHV %+.2f  vs  BIL 0.00'
      % (R['jpst_sharpe'], R['icsh_sharpe'], R['mint_sharpe'], R['shv_sharpe']))
print('the sleeve out-earns bills by ~%.0f bps/yr at ~5x their reward-per-risk...'
      % R['sleeve_bps'])

sleeve pickup over BIL : +49.8 bps/yr  (HAC t = +1.30)
excess-of-BIL Sharpe   : JPST +0.61  ICSH +0.54  MINT +0.40  vs  SHV +0.11  vs  BIL 0.00
the sleeve out-earns bills by ~50 bps/yr at ~5x their reward-per-risk...


## 2. ...but is it *riskless*? The honest stress test

The whole appeal is 'near-riskless'. It isn't. When cash is exactly what you want — a liquidity crunch or a hiking cycle — the sliver of credit+duration bites:

In [2]:
print('March-2020 COVID crunch : BIL %+.2f%%  vs  JPST %+.2f%%  ICSH %+.2f%%  MINT %+.2f%%'
      % (R['covid_bil'], R['covid_jpst'], R['covid_icsh'], R['covid_mint']))
print('2022 rate-hike year     : BIL %+.2f%%  vs  JPST %+.2f%%  ICSH %+.2f%%  MINT %+.2f%%'
      % (R['y2022_bil'], R['y2022_jpst'], R['y2022_icsh'], R['y2022_mint']))
print('max drawdown            : BIL %.2f%%  vs sleeve %.2f%% to %.2f%%'
      % (R['bil_dd'], R['jpst_dd'], R['mint_dd']))

March-2020 COVID crunch : BIL +0.26%  vs  JPST -2.75%  ICSH -3.55%  MINT -4.34%
2022 rate-hike year     : BIL +1.40%  vs  JPST +1.14%  ICSH +0.96%  MINT -1.01%
max drawdown            : BIL -0.21%  vs sleeve -3.28% to -4.62%


## 3. Is the sort just lucky? A live synthetic control

We plant a known pickup in a seeded toy world and check the detector recovers it — and that it stays *silent* on the null (credit tracks cash + a credit factor but with **no** structural carry). No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from ultra_short import data, strategy as st
def pickup_t(planted, seed):
    w = data.synthetic_world(pickup_bps_yr=planted, seed=seed, n_days=2000)
    return st.hac_mean((w['CREDIT'] - w['CASH']).dropna())['t_nw']
null = np.array([pickup_t(0.0, 885+s) for s in range(12)])
plant = np.array([pickup_t(120.0, 885+s) for s in range(12)])
print('null (0 bps/yr), 12 seeds : mean HAC t = %+.2f, |t|>=2 in %d/12' % (null.mean(), int((abs(null)>=2).sum())))
print('planted +120 bps/yr       : mean HAC t = %+.2f, |t|>=2 in %d/12' % (plant.mean(), int((abs(plant)>=2).sum())))

null (0 bps/yr), 12 seeds : mean HAC t = +0.61, |t|>=2 in 0/12
planted +120 bps/yr       : mean HAC t = +3.51, |t|>=2 in 11/12


## 4. The honest verdict

The pickup is **real in the point estimate** — the sleeve out-earns bills by **~50 bps/yr** at an excess Sharpe of **0.62** vs **0.11** for short Treasuries and **0** for bills, and on MINT's full 17-year tape it is **+77 bps/yr at HAC *t* = 2.82**. But it does **not clear the desk's robustness bar**: on the full 3-ETF sleeve the HAC *t* is only **1.30**, the bootstrap Sharpe CI **crosses zero** ([-0.26, +2.21], 9% of resamples negative), and — the killer — the whole edge lives in the **early** window (MINT pre-2018 *t* = 7.44 vs post-2018 *t* = 0.73). And it is **not riskless**: −1% in 2022, −3 to −4% in the COVID crunch, while bills stayed flat. **Signal: Weak. Tradability: Fragile** — costs barely dent it (it's buy-and-hold), but a thin, era-contingent, not-quite-significant carry is not something you can bank.